# Phase 1 — Core RAG Pipeline Test

End-to-end test: **ingest → chunk → embed → ChromaDB → retrieve → Groq answer**.

1. Place your sample PDF in `data/raw/notes/` (or use the included sample TXT).
2. Set `GROQ_API_KEY` in `.env` for full generation.
3. Run cells top to bottom.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from config.settings import COLLECTION_NOTES, GROQ_API_KEY, NOTES_DIR
from src.ingestion.pipeline import IngestionPipeline
from src.vectorstore.chroma_client import ChromaStore

print(f"Project root: {PROJECT_ROOT}")
print(f"Groq key set: {bool(GROQ_API_KEY)}")

In [ ]:
# Change this to your PDF path when ready
SAMPLE_FILE = NOTES_DIR / "sample_os_dbms_cn_oops.txt"
# SAMPLE_FILE = NOTES_DIR / "your_notes.pdf"

assert SAMPLE_FILE.exists(), f"File not found: {SAMPLE_FILE}"
print(f"Using: {SAMPLE_FILE}")

In [ ]:
store = ChromaStore()
store.reset_collection(COLLECTION_NOTES)

pipeline = IngestionPipeline(collection_name=COLLECTION_NOTES, store=store)
n_chunks = pipeline.ingest_file(SAMPLE_FILE)
print(f"Ingested {n_chunks} chunks")

In [ ]:
question = "What is a process in operating systems?"
retrieved = pipeline.retriever.retrieve(question)

for i, chunk in enumerate(retrieved, 1):
    meta = chunk["metadata"]
    print(f"[{i}] {meta['filename']}, p.{meta['page_number']} (dist={chunk['distance']:.4f})")
    print(chunk["text"][:200], "...\n")

In [ ]:
if not GROQ_API_KEY:
    print("Set GROQ_API_KEY in .env to run generation.")
else:
    result = pipeline.answer_question(question)
    print("Answer:", result["answer"])
    print("\nSources:", result["sources"])